# Notebook 6: Explainable AI with Grad-CAM
This notebook illustrates how Grad-CAM extracts gradients and superimposes
colored heatmaps onto raw grayscale MRI inputs to make model classifications transparent.


In [ ]:
import os
from pathlib import Path
import numpy as np
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt
from PIL import Image

sys.path.append('..')
from ml.predict_helper import load_prediction_model, preprocess_single_image
from ml.gradcam import find_last_conv_layer, make_gradcam_heatmap, save_and_display_gradcam

MODELS_DIR = Path('../models')
TEST_DIR = Path('../dataset/processed/test')


### Step 1: Load Model & Test Image
We load our MobileNetV2 model and select a pituitary tumor MRI.


In [ ]:
model = load_prediction_model('MobileNetV2', MODELS_DIR)
img_path = list((TEST_DIR / 'pituitary').glob('*'))[0]
print('Selected Image:', img_path.name)

img_array = preprocess_single_image(str(img_path))
preds = model.predict(img_array)
pred_idx = np.argmax(preds[0])
print(f'Prediction index: {pred_idx} | Confidence: {preds[0][pred_idx]:.4f}')


### Step 2: Generate Grad-CAM Heatmap
We find the last conv layer dynamically and run our gradient calculator.


In [ ]:
last_conv_layer = find_last_conv_layer(model)
print('Last conv layer:', last_conv_layer)

heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer, pred_idx)
print('Heatmap shape:', heatmap.shape)

# Plot raw heatmap
plt.figure(figsize=(4, 4))
plt.imshow(heatmap, cmap='viridis')
plt.title('Raw Activation Heatmap')
plt.colorbar()
plt.show()


### Step 3: Superimpose Heatmap onto Original Image


In [ ]:
output_path = 'gradcam_temp.jpg'
save_and_display_gradcam(str(img_path), heatmap, output_path, alpha=0.4)

# Display side-by-side
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
orig_img = Image.open(img_path)
axes[0].imshow(orig_img, cmap='gray')
axes[0].set_title('Original MRI Scan', fontweight='bold')
axes[0].axis('off')

overlay_img = Image.open(output_path)
axes[1].imshow(overlay_img)
axes[1].set_title('Grad-CAM Overlay', fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Cleanup temporary file
if os.path.exists(output_path):
    os.remove(output_path)
